<a href="https://colab.research.google.com/github/devdasrahul/datavisualisation/blob/main/notebooks/turbofan_eda_and_rul_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predictive Maintenance: Turbofan Engine Degradation Analysis & RUL Baseline
*An exploratory dive into the NASA C-MAPSS dataset.*

In this notebook, I'm taking a step back from the production data pipeline to look directly at the raw telemetry data. Before we build streaming dashboards or automated ETL, we need to understand what the data actually tells us. 

Here, I'll be exploring the engine degradation patterns, engineering a few basic features to extract the signal from the noise, and establishing a baseline Machine Learning model to predict Remaining Useful Life (RUL). This is exactly the kind of exploratory work that informs the transformations we later push into the production pipeline.

In [ ]:
# Setup and Imports
import sys
import os
import urllib.request
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set clean visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Colab: Downloading dataset directly...")
    !pip install -q scikit-learn seaborn pandas matplotlib
    url = "https://raw.githubusercontent.com/prognostic-data/cmapss-mirror/main/CMAPSSData.zip"
    urllib.request.urlretrieve(url, "CMAPSSData.zip")
    with zipfile.ZipFile("CMAPSSData.zip", 'r') as zip_ref:
        zip_ref.extractall("data")
    data_path = "data/train_FD001.txt"
else:
    print("Running locally: Using local data directory...")
    # Adjust this path based on where you run the notebook relative to the project root
    data_path = "../data/raw/train_FD001.txt"
    if not os.path.exists(data_path):
        print(f"Warning: Local data not found at {data_path}. Please download the dataset first.")

### 1. Data Load & Preprocessing
I'm starting with `train_FD001`, which contains run-to-failure trajectories for 100 turbofan engines under a single operating condition. The data doesn't have headers, so I'll map them based on the NASA dataset documentation (3 operational settings, 21 sensor measurements).

In [ ]:
# Define column names based on dataset documentation
columns = ['unit_id', 'cycle', 'setting_1', 'setting_2', 'setting_3'] + [f'sensor_{i}' for i in range(1, 22)]

# Load the raw data
df = pd.read_csv(data_path, sep='\s+', header=None, names=columns)
print(f"Loaded {len(df)} rows of telemetry data.")

# Calculate Remaining Useful Life (RUL) for the training set
# Since this is the training set, every unit runs until failure.
# The RUL at any given cycle is simply (max_cycle_for_unit - current_cycle).
max_cycles = df.groupby('unit_id')['cycle'].max().reset_index()
max_cycles.rename(columns={'cycle': 'max_cycle'}, inplace=True)
df = df.merge(max_cycles, on='unit_id')
df['RUL'] = df['max_cycle'] - df['cycle']
df.drop('max_cycle', axis=1, inplace=True)

df.head()

### 2. Exploratory Analysis

Before throwing algorithms at the data, I need to know what it looks like. 
First, how long do these engines actually last?

In [ ]:
# Distribution of engine lifespans (max cycles)
lifespans = df.groupby('unit_id')['cycle'].max()

plt.figure(figsize=(10, 5))
sns.histplot(lifespans, bins=20, kde=True, color='#0D9488')
plt.title('Distribution of Engine Lifespans (Cycles to Failure)', fontsize=14)
plt.xlabel('Total Cycles', fontsize=12)
plt.ylabel('Number of Engines', fontsize=12)
plt.show()

print(f"Average lifespan: {lifespans.mean():.1f} cycles")
print(f"Shortest lifespan: {lifespans.min()} cycles")
print(f"Longest lifespan: {lifespans.max()} cycles")

There's quite a spread—some engines fail before 150 cycles, while others last well past 300. This tells me that relying on average age alone is a terrible strategy for maintenance. We need to look at the sensors.

Let's look at how the sensors behave as an engine approaches failure. I'll plot a few sensors for a single unit.

In [ ]:
# Plotting sensor trends for Unit 1
unit_1 = df[df['unit_id'] == 1]

sensors_to_plot = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_12']

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
fig.suptitle('Sensor Degradation Trends for Unit 1 (Healthy -> Failure)', fontsize=16)

for ax, sensor in zip(axes.flatten(), sensors_to_plot):
    ax.plot(unit_1['cycle'], unit_1[sensor], color='#4B5563', alpha=0.8)
    ax.set_title(sensor)
    ax.set_ylabel('Reading')
    if ax in axes[2, :]:
         ax.set_xlabel('Cycle (Time)')

plt.tight_layout()
plt.show()

You can visually see the degradation! For example, `sensor_11` (Static pressure at HPC outlet) clearly climbs as the engine wears out, while `sensor_7` drops.

However, not all 21 sensors are useful. Some might be completely flat (no variance) for this specific operating condition (FD001 is a single condition). Let's calculate the variance of each sensor to identify dead signals.

In [ ]:
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
variances = df[sensor_cols].var().sort_values()

plt.figure(figsize=(10, 6))
variances.plot(kind='barh', color='#0D9488')
plt.title('Sensor Variance (Identifying Dead Signals)', fontsize=14)
plt.xlabel('Variance')
plt.xscale('log') # Log scale because some variances are huge and some are exactly 0
plt.show()

dead_sensors = variances[variances == 0.0].index.tolist()
print(f"Sensors with absolutely zero variance: {dead_sensors}")

Those flat sensors (`sensor_1`, `sensor_10`, `sensor_18`, `sensor_19`, etc.) provide zero information about degradation in this dataset. I'll drop them, along with the operational settings which are constant in FD001.

Next, let's look at the correlation matrix. If two sensors move perfectly in sync, we might not need both. More importantly, I want to see which sensors correlate most strongly with the target variable: Remaining Useful Life (`RUL`).

In [ ]:
# Drop dead columns
cols_to_drop = dead_sensors + ['setting_1', 'setting_2', 'setting_3']
df_clean = df.drop(columns=cols_to_drop)
active_sensors = [s for s in sensor_cols if s not in dead_sensors]

# Correlation matrix
corr = df_clean[active_sensors + ['RUL']].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, square=True)
plt.title('Correlation Matrix of Active Sensors and RUL', fontsize=14)
plt.show()

print("Top sensors most negatively correlated with RUL (as engine ages, sensor goes up):")
print(corr['RUL'].sort_values().head(3))
print("\nTop sensors most positively correlated with RUL (as engine ages, sensor goes down):")
print(corr['RUL'].sort_values(ascending=False)[1:4]) # Skip RUL itself

### 3. Feature Engineering Exploration

Raw sensor data is noisy. In the production pipeline, we apply rolling averages to smooth out this noise and expose the underlying degradation trend. Let's demonstrate exactly *why* we do that.

In [ ]:
# Demonstrate rolling average smoothing
window_size = 7
unit_1_smooth = unit_1.copy()
unit_1_smooth['s11_rolling'] = unit_1_smooth['sensor_11'].rolling(window=window_size, min_periods=1).mean()

plt.figure(figsize=(12, 5))
plt.plot(unit_1_smooth['cycle'], unit_1_smooth['sensor_11'], label='Raw Sensor 11', color='#E5E7EB', linewidth=1.5)
plt.plot(unit_1_smooth['cycle'], unit_1_smooth['s11_rolling'], label=f'{window_size}-Cycle Rolling Avg', color='#0D9488', linewidth=3)
plt.title('Why We Use Rolling Averages: Exposing the Trend', fontsize=14)
plt.xlabel('Cycle')
plt.ylabel('Sensor 11 Reading')
plt.legend()
plt.show()

The rolling average makes the true degradation curve obvious. I'll now create a composite "Health Score". Instead of looking at 14 different sensors, what if we combine the most highly correlated ones into a single index?

In [ ]:
# Create a simple Health Index using the most correlated sensors
# We standardize them first so they have equal weight
from sklearn.preprocessing import StandardScaler

top_sensors = ['sensor_11', 'sensor_4', 'sensor_12', 'sensor_7']
scaler = StandardScaler()

# Standardize
df_scaled = df_clean.copy()
df_scaled[top_sensors] = scaler.fit_transform(df_clean[top_sensors])

# For sensors that go down as it breaks (12, 7), we invert them so everything goes UP as it breaks
df_scaled['sensor_12'] = df_scaled['sensor_12'] * -1
df_scaled['sensor_7'] = df_scaled['sensor_7'] * -1

# Combine into a single score
df_clean['composite_health_score'] = df_scaled[top_sensors].mean(axis=1)

# Plot for a few units
plt.figure(figsize=(12, 5))
for unit in [1, 2, 3]:
    u_data = df_clean[df_clean['unit_id'] == unit]
    plt.plot(u_data['cycle'], u_data['composite_health_score'], label=f'Unit {unit}')
    
plt.title('Composite Health Score Trajectory (Multiple Units)', fontsize=14)
plt.xlabel('Cycle')
plt.ylabel('Health Score (Higher = More Degraded)')
plt.legend()
plt.show()

### 4. Baseline RUL Model

Now let's frame the actual machine learning task. 
**Goal:** Given a snapshot of an engine's current sensor readings, predict exactly how many cycles it has left (`RUL`).

I'm going to split the data. I won't just do a random row split, because that would leak future data from the same engine into the training set. Instead, I'll split by `unit_id`. Engines 1-80 will be training, and 81-100 will be held out for testing.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Split by unit
train_units = range(1, 81)
test_units = range(81, 101)

train_df = df_clean[df_clean['unit_id'].isin(train_units)]
test_df = df_clean[df_clean['unit_id'].isin(test_units)]

features = active_sensors + ['composite_health_score']
target = 'RUL'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"Training on {len(X_train)} samples, testing on {len(X_test)} samples.")

I'll train a simple Linear Regression first as a naive baseline, and then a Random Forest Regressor to capture non-linear relationships (since degradation curves are distinctly exponential, not linear).

In [ ]:
# 1. Linear Regression Baseline
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print("--- Linear Regression ---")
print(f"MAE:  {mean_absolute_error(y_test, lr_preds):.2f} cycles")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, lr_preds)):.2f} cycles")

# 2. Random Forest Baseline
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("\n--- Random Forest ---")
print(f"MAE:  {mean_absolute_error(y_test, rf_preds):.2f} cycles")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, rf_preds)):.2f} cycles")

The Random Forest significantly outperforms the linear model. On average, its prediction is off by about ~32 cycles. While that might sound large, remember that early in an engine's life (e.g. cycle 10), it's virtually impossible to predict if it will fail at cycle 180 or 250 because there is no wear yet. The model's error is heavily concentrated on predicting RUL for brand new engines.

Let's look at which sensors the Random Forest actually used to make its decisions.

In [ ]:
# Feature Importances
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importances.tail(10).plot(kind='barh', color='#0D9488')
plt.title('Top 10 Most Important Features (Random Forest)', fontsize=14)
plt.xlabel('Importance Score')
plt.show()

Finally, let's visualize the actual vs. predicted RUL trajectory for a specific engine in the test set to see how the model behaves as the engine approaches failure.

In [ ]:
# Pick a random test unit
test_unit = 91
u91 = test_df[test_df['unit_id'] == test_unit].copy()
u91_preds = rf.predict(u91[features])

plt.figure(figsize=(12, 5))
plt.plot(u91['cycle'], u91['RUL'], label='Actual True RUL', color='#4B5563', linestyle='--')
plt.plot(u91['cycle'], u91_preds, label='Predicted RUL (Random Forest)', color='#0D9488', linewidth=2)
plt.title(f'RUL Prediction Trajectory for Test Unit {test_unit}', fontsize=14)
plt.xlabel('Operating Cycle')
plt.ylabel('Remaining Useful Life (Cycles)')
plt.legend()
plt.show()

Notice how early in the engine's life (cycles 0-50), the prediction is a flat horizontal line? The model realizes that "brand new" engines don't exhibit any wear signals, so it just guesses the fleet average lifespan. As soon as the degradation pattern begins to emerge around cycle 100, the predicted RUL drops sharply and begins accurately tracking the true RUL down to zero.

### 5. Takeaways & Next Steps

**What the data shows:**
1. Turbofan engine degradation is highly non-linear. Engines run perfectly healthy for 60-70% of their lifespan before exhibiting rapid exponential wear.
2. We don't need 21 sensors. Over half of them either have zero variance or are highly redundant. `sensor_11`, `sensor_4`, and `sensor_12` contain the vast majority of the predictive signal.
3. Random Forest handles this non-linear degradation much better than linear models.

**How this connects to the production pipeline:**
In the Streamlit dashboard we built, we use rolling averages and slope calculations to rank engines by degradation severity. This notebook validates *why* that approach works: the degradation slope on the top features is the strongest indicator of impending failure. 

**What a production ML version would need:**
To take this from a notebook baseline to a production ML pipeline, we would need:
- **Piecewise Linear RUL**: Capping the maximum RUL during training (e.g., at 130 cycles) to prevent the model from penalizing itself for poor predictions on brand new engines where wear hasn't started.
- **Sequence Models**: Using LSTMs or 1D-CNNs to natively capture the time-series trajectory, rather than just treating each cycle as an independent snapshot.
- **MLflow Tracking**: Logging experiments, hyperparameter tuning, and model registries.